In [1]:
# algorithme ID3 

import numpy as np
# fonction qui calcule l'entropie initiale 
def entropie(y):
    total = len(y)     # nombre totale dinstances
    classes = set(y)  # récupèration de toutes les classes uniques (oui/non)
    H = 0          # on initialise l'entropie

    for c in classes:    #prcours de chaque classe
        p = y.count(c) / total    # calcule la probabilité de la classe 
        if p > 0:         # éviter log(0)
            H -= p * np.log2(p)     # la formule générale de l'entropie 

    return H                  #retoune de l'entropie
# fonction pour calculer le Gain
def CalculeDeGain(X, y, attribut):
    total = len(y)            # nombre totale de donnee
    valeurs = set([ligne[attribut] for ligne in X])   # recuperation de toutes les valeurs possibles de cet attribut
    H_initiale = entropie(y)                   # entropie avant division
    H_pondére = 0                               # intialisation de l'entropie pondere

    for v in valeurs:                           # on parcourt chaque valeur de l'attribut
        y_v = [y[i] for i in range(len(X)) if X[i][attribut] == v]    # on prend seulement les y correspondants à cette valeur
        poids = len(y_v) / total                 # probabilité d'avoir cette valeur
        H_pondére += poids * entropie(y_v)      # somme pondéré

    return H_initiale - H_pondére        # le gain egale la difference 
 
#fonction pour choisir le meilleur attribut 
def meilleur_attribut(X, y, attributs):
    return max(attributs, key=lambda attr: CalculeDeGain(X, y, attr))  # je prends l'attribut qui donne le gain maximal  

# fonction pour trouver la classe majoritaire
def classe_majoritaire(y):
    return max(set(y), key=y.count)             # je retourne la classe la plus fréquente

# fonction principale ID3
def ID3(X, y, attributs):
    
    if len(set(y)) == 1:    # si toutes les classes identiques
        return y[0]         # je retourne directement la classe

   
    if not attributs:      # s'il n'y a plus d'attributs
        return classe_majoritaire(y)      # je retourne la majorité

    best_attr = meilleur_attribut(X, y, attributs)  # Choisir le meilleur attribut
    arbre = {best_attr: {}}     # sous-arbres ajoutés ensuite selon les valeurs de l'attribut
    valeurs = set([ligne[best_attr] for ligne in X])    # récupèration des valeurs possibles de cet attribut

    for v in valeurs:   # séparer les données selon chaque valeur de l’attribut
        X_v = []        # sous-ensemble des données
        y_v = []       # sous-ensemble des données

        for i in range(len(X)):   # construire les sous-groupes de données associés à chaque valeur d’un attribut
            if X[i][best_attr] == v:
                X_v.append(X[i])
                y_v.append(y[i])

        # si le sous-ensemble est vide
        if not X_v:
            arbre[best_attr][v] = classe_majoritaire(y)
        else:          # sinon supprime l'attribut utilisé
            new_attributs = [a for a in attributs if a != best_attr]   # on enlève l’attribut déjà utilisé pour éviter de le réutilisation
            arbre[best_attr][v] = ID3(X_v, y_v, new_attributs)      # appel récursif de l’algorithme ID3 pour construire le sous-arbre

    return arbre

# fonction pour prédire

def predire(arbre, donnee):

    if not isinstance(arbre, dict):    # si on est sur une feuille
        return arbre                   # retourner la classe

    attribut = list(arbre.keys())[0]    # récupèrer l'attribut courant
    valeur = donnee[attribut]

    if valeur in arbre[attribut]:       # si la valeur observée dans l’exemple correspond à une branche existante
        return predire(arbre[attribut][valeur], donnee)    # on poursuit la prédiction en descendant récursivement dans le sous-arbre
    else:
        return None

# Test du programme

X = [                                                 # données d'entrée
     {"meteo": "soleil", "temperature": "chaud"},
     {"meteo": "soleil", "temperature": "froid"},
     {"meteo": "pluie", "temperature": "froid"},
     {"meteo": "pluie", "temperature": "chaud"}
    ]
y = ["non", "non", "oui", "oui"]                    # classes associées à chaque donnee
attributs = ["meteo", "temperature"]                # liste des attributs utilisés pour construire l'arbre
arbre = ID3(X, y, attributs)                        # construction de l'arbre
print("Arbre :", arbre)

donnee = {"meteo": "soleil", "temperature": "chaud"}
print("Prédiction :", predire(arbre, donnee))

Arbre : {'meteo': {'pluie': 'oui', 'soleil': 'non'}}
Prédiction : non


In [2]:
# algorithme de CART
# fonction qui calcule l'impureté de Fini 

def gini(y):
    total = len(y)     # nombre total d'instances
    classes = set(y)   # on récupère les classes uniques (oui/non)
    g = 1              # on initialise Gini à 1

    for c in classes:                # on parcourt chaque classe
        p = y.count(c) / total       # probabilité de la classe c
        g -= p ** 2                  # on calcule l’impureté de Gini

    return g                         # retourne l'impureté de l'ensemble

# division des données
def split(X, y, attribut, seuil):
    X_gauche, y_gauche = [], []      # exemples dont la valeur est <= seuil
    X_droite, y_droite = [], []      # exemples dont la valeur est > seuil

    for i in range(len(X)):          # on parcourt tous les exemples
        if X[i][attribut] <= seuil:  # si la valeur respecte la condition
            X_gauche.append(X[i])    # on ajoute l'exemple à gauche
            y_gauche.append(y[i])    # et son label
        else:
            X_droite.append(X[i])    # sinon on met à droite
            y_droite.append(y[i])

    return X_gauche, y_gauche, X_droite, y_droite     # retourne les données séparées en deux sous-ensembles

# trouver la meilleure séparation des données
def meilleur_split(X, y, attributs):
    meilleur_gini = float("inf")       # initialisation à +inf pour rechercher le minimum du Gini
    best_attr = None                   # aucun attribut choisi au départ
    best_seuil = None                  # aucun seuil choisi au départ

    for attr in attributs:              # on teste chaque attribut
        valeurs = sorted(set([ligne[attr] for ligne in X]))      # récupérer toutes les valeurs possibles de cet attribut

        for seuil in valeurs:          # on teste chaque valeur comme seuil
            Xg, yg, Xd, yd = split(X, y, attr, seuil)            # on divise les données

            if len(yg) == 0 or len(yd) == 0:                     # si un côté est vide ce split ne sert à rien alors on va l’ignore
                continue         

            # calcul gini pondéré après la séparation
            poids_g = len(yg) / len(y)   # proportion des exemples dans le groupe gauche
            poids_d = len(yd) / len(y)   # proportion des exemples dans le groupe droit

            g = poids_g * gini(yg) + poids_d * gini(yd)         # calcul du Gini total du split

            if g < meilleur_gini:
                meilleur_gini = g       # on met à jour la meilleure valeur de Gini trouvée
                best_attr = attr        # on mémorise l'attribut utilisé pour ce meilleur split
                best_seuil = seuil      # on mémorise le seuil qui donne ce meilleur découpage

    return best_attr, best_seuil        # on retourne la meilleure règle de séparation trouvée


# fonction qui calcule la classe majoritaire
def classe_majoritaire(y):
    return max(set(y), key=y.count)    # retourne la classe la plus fréquente au moment quand on ne peut plus diviser

# construction de l'arbre CART
def CART(X, y, attributs):

    if len(set(y)) == 1:               # si toutes les classes sont identiques alors on retourne directement la valeur de la cible
        return y[0]

    if not attributs:                 # s'il y a plus d’attributs pour diviser on va prend la classe majoritaire           
        return classe_majoritaire(y)

    attr, seuil = meilleur_split(X, y, attributs)                  # on cherche le meilleur attribut et seuil pour diviser

    if attr is None:                   # si aucun split valide trouvé  
        return classe_majoritaire(y)   # retourne la classe majoritaire

    arbre = {f"{attr} <= {seuil}": {}}                              # création du noeud avec la condition

    Xg, yg, Xd, yd = split(X, y, attr, seuil)                        # on divise les données selon ce split
    arbre[f"{attr} <= {seuil}"]["oui"] = CART(Xg, yg, attributs)     # on construit récursivement la branche gauche
    arbre[f"{attr} <= {seuil}"]["non"] = CART(Xd, yd, attributs)     # on construit récursivement la branche droite

    return arbre                      # retourne l’arbre complet

# test
# données d’apprentissage
X = [
    {"age": 20},
    {"age": 25},
    {"age": 35},
    {"age": 40}
]
# classes correspondantes
y = ["oui", "oui", "non", "non"]
attributs = ["age"]                     # attribut utilisé
arbre = CART(X, y, attributs)           # construction de l’arbre
print("Arbre :", arbre)

Arbre : {'age <= 25': {'oui': 'oui', 'non': 'non'}}


In [3]:
# algorithme C4.5


# fonction qui calcule l'entropie
def entropy(y):
    total = len(y)                        # nombre total d'exemples dans y       
    classes = set(y)                      # on récupère les classes uniques (oui/non)
    H = 0                                 # initialisation de l'entropie

    for c in classes:                     # on parcourt chaque classe
        p = y.count(c) / total            # probabilité de la classe c
        H -= p * np.log2(p)               # formule de l'entropie

    return H

# séparation des données
def split(X, y, attribut, seuil):
    Xg, yg, Xd, yd = [], [], [], []       # groupes gauche et droite

    for i in range(len(X)):               # on parcourt toutes les exemples
        if X[i][attribut] <= seuil:       # si la valeur respecte la condition
            Xg.append(X[i])               # exemple envoyé à gauche
            yg.append(y[i])               # label correspondant à gauche
        else:
            Xd.append(X[i])               # sinon exemple envoyé à droite
            yd.append(y[i])               # label correspondant à droite

    return Xg, yg, Xd, yd                 # retourne les deux sous-ensembles

# fonction qui calcule le gain
def CalculeGain(y, yg, yd):
    total = len(y)                        # taille du groupe initial
    H_initiale = entropy(y)               # entropie avant séparation
    H_pondérée = (len(yg)/total)*entropy(yg) + (len(yd)/total)*entropy(yd)     # entropie après séparation

    return H_initiale - H_pondérée

# évalue la qualité de la séparation des données en deux sous-ensembles
def split_info(y, yg, yd):
    total = len(y)                       # nombre total d'exemples

    def term(size):                      # fonction auxiliaire
        if size == 0:                    # éviter log(0)
            return 0
        p = size / total                 # proportion du sous-groupe
        return -p * np.log2(p)           # contribution à la mesure

    return term(len(yg)) + term(len(yd)) # somme des deux parties

# gain Ratio 
def gain_ratio(y, yg, yd):
    CG = CalculeGain(y, yg, yd)
    si = split_info(y, yg, yd)

    if si == 0:                          # éviter division par zéro
        return 0

    return ig / si                       # normalisation du gain

# meilleur split
def meilleur_split(X, y, attributs):
    best_ratio = -1                      # valeur initiale minimale pour pouvoir trouver un ratio plus grand
    best_attr = None                     # aucun attribut sélectionné au départ
    best_seuil = None                    # aucun seuil sélectionné au départ

    for attr in attributs:               # on teste chaque attribut
        valeurs = sorted(set([x[attr] for x in X]))                 # toutes les valeurs possibles

        for seuil in valeurs:            # on teste chaque valeur comme seuil
            Xg, yg, Xd, yd = split(X, y, attr, seuil)               # division des données

            if len(yg) == 0 or len(yd) == 0:                        # split invalide
                continue                 # on passe au suivant

            ratio = gain_ratio(y, yg, yd)                           # évaluation du split

            if ratio > best_ratio:       # si meilleur que l'ancien
                best_ratio = ratio       # mise à jour du score
                best_attr = attr         # mémorisation de l'attribut 
                best_seuil = seuil       # mémorisation du seuil

    return best_attr, best_seuil        # retourne le meilleur choix


# fonction qui classe majoritaire
def classe_majoritaire(y):
    return max(set(y), key=y.count)

# construction de l'arbre C4.5
def C45(X, y, attributs):

    if len(set(y)) == 1:               # si toutes les classes sont identiques
        return y[0]                    # on retourne directement la classe

    if not attributs:                  # s'il n'y a plus d'attributs
        return classe_majoritaire(y)   # on prend la classe dominante

    attr, seuil = meilleur_split(X, y, attributs)                    # meilleur split

    if attr is None:                   # si aucun split valide trouvé
        return classe_majoritaire(y)   # on retourne la majorité

    arbre = {f"{attr} <= {seuil}": {}} # création du nœud de décision
    Xg, yg, Xd, yd = split(X, y, attr, seuil)                       # séparation des données
    arbre[f"{attr} <= {seuil}"]["oui"] = C45(Xg, yg, attributs)     # construction récursive de la branche gauche
    arbre[f"{attr} <= {seuil}"]["non"] = C45(Xd, yd, attributs)     # construction récursive de la branche droite

    return arbre                       # retourne l'arbre complet



In [4]:
#algorithme DBSCAN

# fonction qui calcule la distance euclidienne
def distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)  # formule de distance entre deux points

# recherche des voisins
# data : dataset , point : index du point , eps : distance maximale entre deux point
def voisins(data, point, eps):

    voisins_proches = []                                    # liste des voisins trouvés
    for i in range(len(data)):                              # parcourir tous les points du dataset
        d = distance(data[point], data[i])                  # calcul de la distance entre deux points

        if d <= eps:                                        # si la distance est inférieure à eps
            voisins_proches.append(i)                       # ajout de l'index du point à la liste des voisins

    return voisins_proches                                  # retourne les voisins du point

# DBSCAN
# min_pts : minimum de voisins 
def DBSCAN(data, eps, min_pts):

    cluster_id = 0                                         # compteur des clusters
    labels = [-1] * len(data)                            #contient les clusters et -1 signifie que le point non encore traité
    for point in range(len(data)):                       # parcourir tous les points

        
        if labels[point] != -1:                        # ignorer les points déjà traités
            continue

        voisins_point = voisins(data, point, eps)      # récupérer les voisins du point courant
        if len(voisins_point) < min_pts:              # vérifier si le point est isolé
            labels[point] = 0                         # 0 signifie que le point est un point bruit c-a-d isolé
            
        else:  
            cluster_id += 1                         # nouveau cluster trouvé  
            labels[point] = cluster_id             # créer le cluster
            file = voisins_point.copy()            # liste des voisins à explorer

            while file:                           # explorer récursivement les voisins tant que le file n'est pas vide

                voisin = file.pop(0)              # retirer le premier voisin de la file pour le traiter
                if labels[voisin] == 0:           # si le point était du bruit
                    labels[voisin] = cluster_id   # il est maintenant ajouté au cluster actuel

                if labels[voisin] != -1:          # ignorer les points déjà traiter
                    continue

                labels[voisin] = cluster_id      # ajouter le point au cluster
                nouveaux_voisins = voisins(data, voisin, eps)    # récupérer les voisins du voisin

                if len(nouveaux_voisins) >= min_pts:             # si densité suffisante
                    file.extend(nouveaux_voisins)                 # ajouter les voisins à explorer

    return labels                                               # retourne le cluster de chaque point

# TEST

data = [
    [1, 1],
    [1.5, 2],
    [2, 1],

    [8, 8],
    [8, 9],
    [9, 8],

    [20, 20]
]

eps = 2   # rayon des voisins
min_pts = 2 # minimum de voisins pour créer un cluster

resultat = DBSCAN(data, eps, min_pts)

print("Clusters :", resultat)

Clusters : [1, 1, 1, 2, 2, 2, 0]


In [5]:
# algorithme naive bayes

# Entraînement
def entrainer(X, y):

    modele = {}                       # dictionnaire qui va stocker toutes les probabilités
    classes = set(y)                  # récupérer toutes les classes
    for c in classes:                 # parcourir chaque classe
        lignes_classe = []            # liste des lignes appartenant à la classe actuelle

        for i in range(len(y)):       # parcourir toutes les lignes du dataset
            if y[i] == c:             # vérifier si la ligne appartient à la classe actuelle
                lignes_classe.append(X[i])               # ajouter cette ligne dans lignes_classe
        prior = len(lignes_classe) / len(y)              # calcul de la probabilité de la classe
        modele[c] = {                 # créer un espace pour stocker les informations de la classe
            "prior": prior,            # stocker la probabilité initiale de la classe
            "attributs": {}            # stocker les probabilités des attributs
        }
        attributs = X[0].keys()       # récupérer les noms des attributs

        for attr in attributs:         # parcourir chaque attribut
            modele[c]["attributs"][attr] = {}             # parcourir chaque attribut
            valeurs = set([ligne[attr] for ligne in X])   # récupérer toutes les valeurs possibles

            for v in valeurs:          # parcourir chaque valeur possible
                count = 0             # compteur des occurrences

                for ligne in lignes_classe:               # parcourir les lignes de la classe actuelle
                    if ligne[attr] == v:                   # vérifier si la valeur existe dans cette ligne
                        count += 1     # augmenter le compteur
                proba = (count + 1) / (len(lignes_classe) + len(valeurs))   # calcul de la probabilité avec correction de Laplace
                modele[c]["attributs"][attr][v] = proba    # stocker la probabilité dans le modèle

    return modele                     # retourner le modèle entraîné

# Prédiction
def predire(modele, exemple):

    meilleurscore = -1               # variable qui garde le score le plus élevé
    meilleureclasse = None           # variable qui contiendra la classe finale prédite
    for c in modele:                # parcourir chaque classe du modèle
        score = modele[c]["prior"]  # commencer le score avec le prior de la classe
        for attr in exemple:        # parcourir les attributs
            valeur = exemple[attr]  # récupérer la valeur de l'attribut
            if valeur in modele[c]["attributs"][attr]:                    # vérifier si la valeur existe dans le modèle
                score *= modele[c]["attributs"][attr][valeur]             # multiplier le score par la probabilité correspondante
        if score > meilleurscore:     # vérifier si le score actuel est meilleur
            meilleurscore = score      # mettre à jour le meilleur score
            meilleureclasse = c       # sauvegarder la meilleure classe

    return meilleureclasse            # retourner la classe prédite

X = [                                # dataset d'entraînement
    {"meteo": "soleil", "vent": "faible"},
    {"meteo": "soleil", "vent": "fort"},
    {"meteo": "pluie", "vent": "faible"},
    {"meteo": "pluie", "vent": "fort"}
]

y = ["oui", "oui", "non", "non"]    # classes correspondantes
modele = entrainer(X, y)            # apprentissage des probabilités
exemple = {"meteo": "soleil", "vent": "faible"}  # nouvel exemple

resultat = predire(modele, exemple) # prédiction de la class

print("Classe prédite :", resultat) # affichage

Classe prédite : oui


In [6]:
#algorithme de Q-learning
import random           # importer les fonctions permettant de faire des choix aléatoires

# environnement
# matrice des récompenses
#(-1 : action interdite, 0 : deplacement normale, 100 : objectif atteint)
R = [
    [-1, 0, -1],        # état 0
    [0, -1, 100],        # état 1
    [-1, 0, 100]        # état 2 (objectif)
]

# Initialisation Q-table
# création de la table Q remplie par des zéros
Q = [                    #représente à quel point cette action est bonne dans cet état
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0]
]
# paramètres
# vitesse d’apprentissage
alpha = 0.8                
gamma = 0.9                            # facteur de réduction qui permet à l’agent de prendre en compte le futur
episodes = 100                         # nombre d’itérations : combien de fois l’agent va répéter l’apprentissage

# entraînement
for episode in range(episodes):

    etat = random.randint(0, 2)        # choisir un état de départ aléatoire entre 0 et 2

    while etat != 2:                  # continuer tant qu’on n’atteint pas l’objectif
        actions_possibles = []        # liste des actions possibles
        for action in range(len(R[etat])):          # parcourir les actions de l’état courant
            if R[etat][action] != -1:               # vérifier si l’action est autorisée car -1 est un deplacemement interdit
                actions_possibles.append(action)    # ajouter l’action à la liste

        action = random.choice(actions_possibles)  # choisir une action aléatoire parmi les actions autorisées
        recompense = R[etat][action]               # récupérer la récompense obtenue après cette action
        max_q = max(Q[action])        # récupérer la meilleure récompense future possible depuis le nouvel etat
        Q[etat][action] = Q[etat][action] + alpha * (               # appliquer la formule Q-learning
            recompense + gamma * max_q - Q[etat][action]
        )
        etat = action                 # passer au nouvel état

# affichage
print("Q-table finale :")
for ligne in Q:
    print(ligne)

Q-table finale :
[0, 90.0, 0]
[81.0, 0, 100.0]
[0, 0, 0]


In [7]:
# algorithme de PCA (Principal Component Analysis)
# Standardisation des données
def standardiser(X):
    moyenne = np.mean(X, axis=0)             # calcul de la moyenne de chaque colonne (feature)
    ecart_type = np.std(X, axis=0)          # calcul de l’écart-type de chaque colonne
    return (X - moyenne) / ecart_type       # normalisation : centrer et réduire les données

# PCA from scratch
# X = données, k = nombre de dimensions voulues
def PCA(X, k):
    X_centree = X - np.mean(X, axis=0)      # centrer les données (moyenne = 0)
    covariance = np.cov(X_centree, rowvar=False)                       # mesure les relations entre les variables
    valeurs_propres, vecteurs_propres = np.linalg.eig(covariance)      # trouve les directions principales des données ( valeurs_propres :l’importance de chaque direction , vecteurs_propres : les directions dans lesquelles les données varient) 
    indices = np.argsort(valeurs_propres)[::-1]                        # trie les valeurs propres de la plus grande à la plus petite
    vecteurs_propres = vecteurs_propres[:, indices]                    # réorganiser les vecteurs propres selon l’importance
    composants = vecteurs_propres[:, :k]                               # sélectionner les k directions principales
    X_reduit = np.dot(X_centree, composants)                           # projeter les données dans les nouvelles directions principales

    return X_reduit                        # retourner les données transformées

# Exemple de test
# dataset simple 
X = np.array([
    [2.5, 2.4],
    [0.5, 0.7],
    [2.2, 2.9],
    [1.9, 2.2],
    [3.1, 3.0],
    [2.3, 2.7]
])
X_reduit = PCA(X, 1)                    # réduction de dimension 2D vers 1D
print("Données après PCA :")            # affichage du résultat
print(X_reduit)

Données après PCA :
[[ 0.35725229]
 [-2.26208195]
 [ 0.48967143]
 [-0.21285398]
 [ 1.2056734 ]
 [ 0.42233881]]


In [8]:
# algorithme XGBoost
# Dataset
# données d'entrée
X = np.array([20, 30, 40, 50])
y = np.array([100, 150, 200, 250])                  # vraies valeurs à prédire

# Première prédiction
prediction = np.mean(y)                            # calcul de la moyenne des valeurs réelles
predictions = np.full(len(y), prediction)          # créer une liste contenant la première prédiction
print("Première prédiction :")                     # afficher la prédiction initiale
print(predictions)

# Calcul des erreurs
residus = y - predictions                         # calcul des résidus (erreurs)
print("\nRésidus :")                              # afficher les erreurs
print(residus)

# Petit arbre correctif
corrections = []                                # tableau qui stockera les corrections
for r in residus:                               # parcourir chaque résidu
    correction = 0.5 * r                        # calcul d'une petite correction
    corrections.append(correction)              # ajouter la correction
corrections = np.array(corrections)             # convertir en tableau numpy
print("\nCorrections :")                        # afficher les corrections
print(corrections)

# Mise à jour des prédictions
predictions = predictions + corrections         # ajouter les corrections aux anciennes prédictions
print("\nNouvelles prédictions :")              # afficher les nouvelles prédictions
print(predictions)

# Nouvelle erreur
nouveaux_residus = y - predictions             # recalculer les erreurs
print("\nNouveaux résidus :")                  # afficher les nouvelles erreurs
print(nouveaux_residus)

Première prédiction :
[175. 175. 175. 175.]

Résidus :
[-75. -25.  25.  75.]

Corrections :
[-37.5 -12.5  12.5  37.5]

Nouvelles prédictions :
[137.5 162.5 187.5 212.5]

Nouveaux résidus :
[-37.5 -12.5  12.5  37.5]
